In [ ]:
using ITensors
using ITensorMPS
using Random

using LinearAlgebra
using Statistics
#=
    Generates the MPO for the EHM Hamiltonian 
    with strengths J, U and V. 
    Requires a SiteType sites.
=#
function H_EHM(N, J, U, V, sites)
    os = OpSum()
    for i in 1:(N - 1)
      # Knetic 
      os -= J, "Cdagup", i, "Cup", i + 1
      os -= J, "Cdagup", i + 1, "Cup", i
      os -= J, "Cdagdn", i, "Cdn", i + 1
      os -= J, "Cdagdn", i + 1, "Cdn", i
      # Nearest-neighbours
      os += V, "Ntot", i, "Ntot", i + 1
    end
    # on-site
    for i in 1:N
      os += U, "Nupdn", i
    end
    return MPO(os, sites)
end

function random_metallic_state(L, Nup, Ndn)
    state = fill("Emp", L)
    p = Nup + Ndn
    for i in 1:L
        j = L - i
        if(p > j)
            state[j] = "UpDn"
            p -= 2
        elseif (p > 0) 
            state[j] = j % 2 == 1 ? "Up" : "Dn"
            p -= 1
        end
    end
    return state
end

function average_single_site_entanglement(L, up, dn, updn)
    single_site_entanglement = fill(0.0, L)

    for i in 1:L 
        w_2 = updn[i] 
        w_up = up[i] - w_2 
        w_dn = dn[i] - w_2 
        w_0 = 1 - w_up - w_dn - w_2 
        single_site_entanglement[i] = 1 - (w_2^2 + w_up^2 + w_dn^2 + w_0^2)
    end
    return Statistics.mean(single_site_entanglement)
end
#=
    Returns the density of up and down 
    electrons.
=#
function density_operators(N, psi)
    upd = fill(0.0, N)
    dnd = fill(0.0, N)
    updn = fill(0.0, N) 
    for j in 1:N
    orthogonalize!(psi, j)
    psidag_j = dag(prime(psi[j], "Site"))
    upd[j] = scalar(psidag_j * op(sites, "Nup", j) * psi[j])
    dnd[j] = scalar(psidag_j * op(sites, "Ndn", j) * psi[j])
    updn[j] = scalar(psidag_j * op(sites, "Nupdn", j) * psi[j])
    end
    return upd, dnd, updn
end

function Ep(rdm, L)
    println("trace (rho) = ", tr(rdm))
    println("ishermitian(rho) = ", ishermitian(rdm))
    lambdas = eigvals(rdm)
    S = 0
    println("eigenvals = ", lambdas)
    for lambda in lambdas
        lambda = real(lambda) 
        # S -= lambda * log2(lambda)
        # CRITICAL: Handle lambda <= 0 for log2
        if lambda > 1e-15 # A small threshold to avoid errors with log2
            S -= lambda * log2(lambda)
        end
    end 
    return S 
end

function build_1_particle_rdm(psi) 
    L = length(psi)
    #=  
        N and not L since any site can have spin up or down
    =#
    rho_1 = zeros(ComplexF64, 2*L, 2*L) 

    Cupup = correlation_matrix(psi, "Cdagup", "Cup")
    Cdndn = correlation_matrix(psi, "Cdagdn", "Cdn")
    Cupdn = correlation_matrix(psi, "Cdagup", "Cdn")
    Cdnup = correlation_matrix(psi, "Cdagdn", "Cup")   

    for i in 1:L
        for j in 1:L
            # Blocks of the correlation matrix.
            i_up = 2 * (i-1) + 1 
            i_dn = 2 * (i-1) + 2
            j_up = 2 * (j-1) + 1
            j_dn = 2 * (j-1) + 2

            rho_1[i_up, j_up] = Cupup[i,j]
            rho_1[i_up, j_dn] = Cupdn[i,j]
            rho_1[i_dn, j_up] = Cdnup[i,j]
            rho_1[i_dn, j_dn] = Cdndn[i,j]
        end 
    end
    rho_1 = rho_1 / L
end

build_1_particle_rdm (generic function with 1 method)

generating input range of values for $U$ and $V$.

In [25]:
results = "../results/EHM_Itensor_Phase_Diagram/"

U_max = 6.0
V_max = 4

U_min = -U_max
V_min = -V_max

N_Points = 50
L = 6
#=
        
        L = 6 e 7 
        N_Points = 100

        L = 5 e 6 
        N_Points = 50

=#
using DelimitedFiles 

U_values = range(U_min, stop=U_max, length=N_Points)

filename = joinpath(results, "U_vals_NPoints=$(N_Points).txt")
writedlm(filename, U_values)

V_values = range(V_min, stop=V_max, length=N_Points)

filename = joinpath(results, "V_vals_NPoints=$(N_Points).txt")
writedlm(filename, V_values)

In [11]:
L = 7

sites = siteinds("Electron", L; conserve_qns=true)

maxdim = [50, 100, 200, 400, 800, 800]
cutoff = [1E-14]

nsweeps = 10

Npart = floor(Int, L/2) 
Nup = Npart + L % 2 
Ndn = L - Nup 

state = random_metallic_state(L, Nup, Ndn)

println(state)

psi0 = random_mps(sites, state; linkdims=10)

U = 0.5 
V = -7.5
J = 1.0

H = H_EHM(L, J, U, V, sites)

energy, psi = dmrg(H, psi0; nsweeps, maxdim, cutoff)

["Up", "Dn", "Up", "Dn", "Up", "UpDn", "Emp"]
After sweep 1 energy=-73.91579547166529  maxlinkdim=50 maxerr=7.01E-13 time=11.654
After sweep 2 energy=-73.91644783115903  maxlinkdim=50 maxerr=9.79E-15 time=0.110
After sweep 3 energy=-73.91651811761197  maxlinkdim=51 maxerr=9.93E-15 time=0.118
After sweep 4 energy=-73.91658823150406  maxlinkdim=50 maxerr=9.18E-15 time=0.105
After sweep 5 energy=-73.91666775964455  maxlinkdim=53 maxerr=9.78E-15 time=0.113
After sweep 6 energy=-73.91675016776442  maxlinkdim=53 maxerr=8.69E-15 time=0.108
After sweep 7 energy=-73.91682951914609  maxlinkdim=54 maxerr=9.92E-15 time=0.113
After sweep 8 energy=-73.91689991878813  maxlinkdim=54 maxerr=8.78E-15 time=0.107
After sweep 9 energy=-73.91696880917671  maxlinkdim=54 maxerr=9.68E-15 time=0.115
After sweep 10 energy=-73.91703193278188  maxlinkdim=55 maxerr=8.32E-15 time=0.105


(-73.91703193278188, MPS
[1] ((dim=4|id=276|"Electron,Site,n=1") <Out>
 1: QN(("Nf",0,-1),("Sz",0)) => 1
 2: QN(("Nf",1,-1),("Sz",1)) => 1
 3: QN(("Nf",1,-1),("Sz",-1)) => 1
 4: QN(("Nf",2,-1),("Sz",0)) => 1, (dim=4|id=166|"Link,l=1") <Out>
 1: QN(("Nf",5,-1),("Sz",1)) => 1
 2: QN(("Nf",6,-1),("Sz",0)) => 1
 3: QN(("Nf",6,-1),("Sz",2)) => 1
 4: QN(("Nf",7,-1),("Sz",1)) => 1)
[2] ((dim=16|id=657|"Link,l=2") <Out>
 1: QN(("Nf",3,-1),("Sz",1)) => 1
 2: QN(("Nf",4,-1),("Sz",0)) => 2
 3: QN(("Nf",4,-1),("Sz",2)) => 2
 4: QN(("Nf",5,-1),("Sz",-1)) => 1
 5: QN(("Nf",5,-1),("Sz",1)) => 4
 6: QN(("Nf",5,-1),("Sz",3)) => 1
 7: QN(("Nf",6,-1),("Sz",0)) => 2
 8: QN(("Nf",6,-1),("Sz",2)) => 2
 9: QN(("Nf",7,-1),("Sz",1)) => 1, (dim=4|id=900|"Electron,Site,n=2") <Out>
 1: QN(("Nf",0,-1),("Sz",0)) => 1
 2: QN(("Nf",1,-1),("Sz",1)) => 1
 3: QN(("Nf",1,-1),("Sz",-1)) => 1
 4: QN(("Nf",2,-1),("Sz",0)) => 1, (dim=4|id=166|"Link,l=1") <In>
 1: QN(("Nf",5,-1),("Sz",1)) => 1
 2: QN(("Nf",6,-1),("Sz",0)) => 

Resultado por ED: $\mathcal{L} = 0.7148148148148148$


In [12]:
rho_1 = build_1_particle_rdm(psi)

14×14 Matrix{ComplexF64}:
   2.88319e-5+0.0im          0.0+0.0im  …           0.0+0.0im
          0.0+0.0im   1.28906e-5+0.0im       1.23704e-5+0.0im
  0.000421257+0.0im          0.0+0.0im              0.0+0.0im
          0.0+0.0im  0.000186619+0.0im       0.00017873+0.0im
   0.00128668+0.0im          0.0+0.0im              0.0+0.0im
          0.0+0.0im   0.00132145+0.0im  …    0.00124199+0.0im
  -7.86292e-5+0.0im          0.0+0.0im              0.0+0.0im
          0.0+0.0im  -9.45435e-5+0.0im     -0.000146786+0.0im
 -0.000195859+0.0im          0.0+0.0im              0.0+0.0im
          0.0+0.0im   3.28558e-5+0.0im      0.000680919+0.0im
   0.00147135+0.0im          0.0+0.0im  …           0.0+0.0im
          0.0+0.0im   9.11453e-5+0.0im      0.000132246+0.0im
  0.000100435+0.0im          0.0+0.0im              0.0+0.0im
          0.0+0.0im   1.23704e-5+0.0im       1.49755e-5+0.0im

In [13]:
E_p = Ep(rho_1, L) - log2(L)
println("E_p = ", E_p)

trace (rho) = 1.0000000000000002 + 0.0im
ishermitian(rho) = true
eigenvals = [3.1327791725575666e-8, 4.6175822766170324e-8, 2.072764869566689e-7, 6.094978964792258e-6, 6.3487889528316956e-6, 7.661571529645532e-5, 7.686618581729431e-5, 0.14278306597943932, 0.14278818146993816, 0.14284319046205818, 0.14284821331789826, 0.1428569928674934, 0.1428570179710359, 0.14285712748300414]
E_p = 0.002092878374889917


In [14]:
  N = 8
  m = 4

  s = siteinds("Electron", N; conserve_qns=true)
  psi = random_mps(s, n -> isodd(n) ? "Up" : "Dn"; linkdims=m)
  
  Cuu = correlation_matrix(psi, "Cdagup", "Cup")

8×8 Matrix{Float64}:
  0.808818      0.245079     -0.0418417   …   0.000398158   4.51335e-5
  0.245079      0.344283      0.0195619      -0.00169821   -0.000192502
 -0.0418417     0.0195619     0.154423       -0.00442554   -0.00050166
 -0.00784911    0.0182162     0.0576614      -0.00629708   -0.00071381
 -0.0200605    -0.00206634    0.053123        0.00106032    0.000120194
 -0.00483715    0.0206312     0.0537651   …   0.000972718   0.000110263
  0.000398158  -0.00169821   -0.00442554      0.140562      0.110831
  4.51335e-5   -0.000192502  -0.00050166      0.110831      0.696444

- trying to create a non-uniform mesh grid to compute the phase diagram.

In [15]:
# param1_segment1 = range(0.0, stop=0.9, length=10)
# param1_segment2 = range(0.91, stop=1.09, length=40)
# param1_segment3 = range(1.1, stop=2.0, length=10)
# param1_values = vcat(collect(param1_segment1), collect(param1_segment2), collect(param1_segment3))
# param2_values = collect(range(0.0, stop=1.0, length=50))

In [16]:
N = 2
m = 4

s = siteinds("Electron", N; conserve_qns=true)

psi = productMPS(s, ["Up", "Dn"])

MPS
[1] ((dim=4|id=374|"Electron,Site,n=1") <Out>
 1: QN(("Nf",0,-1),("Sz",0)) => 1
 2: QN(("Nf",1,-1),("Sz",1)) => 1
 3: QN(("Nf",1,-1),("Sz",-1)) => 1
 4: QN(("Nf",2,-1),("Sz",0)) => 1, (dim=1|id=459|"Link,l=1") <In>
 1: QN(("Nf",1,-1),("Sz",1)) => 1)
[2] ((dim=1|id=459|"Link,l=1") <Out>
 1: QN(("Nf",1,-1),("Sz",1)) => 1, (dim=4|id=488|"Electron,Site,n=2") <Out>
 1: QN(("Nf",0,-1),("Sz",0)) => 1
 2: QN(("Nf",1,-1),("Sz",1)) => 1
 3: QN(("Nf",1,-1),("Sz",-1)) => 1
 4: QN(("Nf",2,-1),("Sz",0)) => 1)


In [17]:
rho_1 = build_1_particle_rdm(psi)

println(tr(rho_1))
println("is hermitian ? ", ishermitian(rho_1))

1.0 + 0.0im
is hermitian ? true


In [18]:
real(rho_1)

4×4 Matrix{Float64}:
 0.5  0.0  0.0  0.0
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.5

In [19]:
println(size(rho_1))

(4, 4)


In [20]:
Cupup = correlation_matrix(psi, "Cdagup", "Cup")
Cdndn = correlation_matrix(psi, "Cdagdn", "Cdn")
Cupdn = correlation_matrix(psi, "Cdagup", "Cdn")
Cdnup = correlation_matrix(psi, "Cdagdn", "Cup")

println("Cupup = ", Cupup)
println("Cupdn = ", Cupdn)
println("Cdnup = ", Cdnup)
println("Cdndn = ", Cdndn)

Cupup = [1.0 0.0; 0.0 0.0]
Cupdn = [0.0 0.0; 0.0 0.0]
Cdnup = [0.0 0.0; 0.0 0.0]
Cdndn = [0.0 0.0; 0.0 1.0]


In [21]:
E_p = Ep(rho_1, N) - log2(N)
println("E_p = ", E_p)

trace (rho) = 1.0 + 0.0im
ishermitian(rho) = true
eigenvals = [0.0, 0.0, 0.5, 0.5]
E_p = 0.0


2-rdm

Previous result $E_p = -1.893297478846248$ for $(U, V) = (-6.0, -4.0)$